In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

PROJECT_ROOT = Path("/Users/yimingzang/Documents/Project/benchmark2")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import bayesflow as bf
pd.set_option("display.max_columns", 100)

In [ ]:
from benchmark.examples.diffusion.config import MODELS, RESULT_DIR, TrainingConfig, ensure_dirs
from benchmark.examples.diffusion.dataset import wagenmakers
from benchmark.examples.diffusion.simulators import SIMULATORS
from benchmark.examples.diffusion.approximators.indirect import load_approximator, load_history
from benchmark.examples.diffusion.results.results import (
    attach_gold_standard,
    estimate_model_comparison,
    load_results,
    save_results,
)
from benchmark.examples.diffusion.results.summary_diagnostic import (
    add_summary_diagnostics,
    fit_references,
    load_diagnostic,
    load_references,
    save_diagnostic,
    save_references,
)
from benchmark.examples.diffusion.results.plots import (
    plot_logml_error_vs_rho,
    plot_pmp_error_vs_log_ambiguity,
    plot_pmp_error_vs_rho,
    plot_pmp_estimates_vs_distance,
)

ensure_dirs()


In [ ]:
from tqdm.auto import tqdm as original_tqdm
import bayesflow.approximators.helpers.samplers as bf_samplers
import bayesflow.approximators.helpers.conditions as bf_conditions

def quiet_tqdm(*args, **kwargs):
    kwargs["disable"] = True
    return original_tqdm(*args, **kwargs)

bf_samplers.tqdm = quiet_tqdm
bf_conditions.tqdm = quiet_tqdm


In [ ]:
config = TrainingConfig(epochs=50, batch_size=64, num_batches=64, summary_multiplier=1)
summary_tag = config.summary_label
metric = "linf"
results_path = RESULT_DIR / f"npe_{summary_tag}_results_gold.csv"
references_path = RESULT_DIR / f"npe_{summary_tag}_references_{metric}.pkl"
diagnostic_path = RESULT_DIR / f"npe_{summary_tag}_empirical_diagnostic_{metric}.csv"

approximators = {model: load_approximator(model, config=config) for model in MODELS}
histories = {model: load_history(model, config=config) for model in MODELS}


In [ ]:
empirical = wagenmakers.df_array
ids = wagenmakers.ids

empirical.shape, ids[:5]

In [ ]:
results = estimate_model_comparison(
    approximators,
    empirical,
    ids=ids,
    dataset="empirical",
    num_samples=2048,
    batch_size=8,
)

results_gold = attach_gold_standard(results)
save_results(results_gold, results_path)
results_gold.head()


In [ ]:
results_gold = load_results(results_path)

In [ ]:
references = fit_references(
    approximators,
    SIMULATORS,
    n_fit=2000,
    n_calibration=2000,
    alpha=0.05,
    metric=metric,
    n_boot=1000,
)

save_references(references, references_path)
{model: {key: references[model][key] for key in ["summary_dim", "median", "low", "high"]} for model in references}


## 5. Add summary diagnostics to empirical results

In [ ]:
diagnostic = add_summary_diagnostics(results_gold, empirical, approximators, references)
save_diagnostic(diagnostic, diagnostic_path)

display_columns = (
    ["id"]
    + [f"rho_{model}" for model in MODELS]
    + [f"regime_{model}" for model in MODELS]
    + ["summary_ambiguity"]
    + [f"pmp_{model}" for model in MODELS]
    + [f"gold_pmp_{model}" for model in MODELS]
    + [f"abs_pmp_error_{model}" for model in MODELS]
)
diagnostic[display_columns].head(20)


## 6. Load cached tables for plotting

After the computations above have been saved once, you can restart the kernel and run this cell before plotting.


In [ ]:
# Uncomment these lines to plot from saved results without recomputing logML or reference distances.
# results_gold = load_results(results_path)
# references = load_references(references_path)
# diagnostic = load_diagnostic(diagnostic_path)


## 7. Basic plots


In [ ]:
plot_logml_error_vs_rho(diagnostic, filename=f"npe_logml_error_vs_rho_{summary_tag}_{metric}.png");
plot_pmp_error_vs_rho(diagnostic, filename=f"npe_pmp_error_vs_rho_{summary_tag}_{metric}.png");
plot_pmp_error_vs_log_ambiguity(diagnostic, filename=f"npe_pmp_error_vs_log_ambiguity_{summary_tag}_{metric}.png");
